# RF-IDS — Supervised Random Forest

Trains on all of `data/processed/train.csv` (benign + attack). To keep the comparison with the Baseline autoencoder fair, this does NOT use RF's default 0.5 probability cutoff — it picks a threshold from benign `val` scores at the same target FPR as the Baseline (see `src/evaluate.py` / `src/rf_ids.py` docstrings for why).

This notebook assumes `data/processed/{train,val,test}.csv` already exist — run `Milestone1_Draft.ipynb` first if they don't, or if you've changed the data split.

In [1]:
import sys
from pathlib import Path

# Make the project root importable regardless of PyCharm's working directory
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/skishore/Documents/GitHub/CSCI6330-Detector-Project


## Train, threshold, and evaluate on val

`run_pilot()` does all of it: trains on the full train split (benign + attack), picks a matched-FPR threshold on val-benign scores (1% by default, same as the Baseline), evaluates on the full val split, and saves the model/threshold to `models/` for reuse.

In [2]:
from src.rf_ids import run_pilot

rf, threshold, val_metrics = run_pilot(
    processed_dir=PROJECT_ROOT / "data" / "processed",
    target_fpr=0.01,
    models_dir=PROJECT_ROOT / "models",
)

  dropped 465 rows with inf/NaN feature values
  dropped 85 rows with inf/NaN feature values
Training on 779176 rows (135627 attack, 643549 benign)

Threshold set at 1.0% target FPR on benign val scores (matched to Baseline's protocol, not RF's default 0.5 cutoff): 0.005000

=== RF-IDS — val split (pilot checkpoint) ===
  threshold : 0.0050
  accuracy  : 0.9924
  precision : 0.9581
  recall    : 1.0000
  f1        : 0.9786
  fpr       : 0.0092
  fnr       : 0.0000
  n         : 166978
  n_attack  : 29061
  n_benign  : 137917

Per-attack recall (val):
                  recall      n
label                          
DoS GoldenEye        1.0   1542
DoS Hulk             1.0  25927
DoS Slowhttptest     1.0    784
DoS slowloris        1.0    807
Heartbleed           1.0      1
Saved RF-IDS model and threshold to /Users/skishore/Documents/GitHub/CSCI6330-Detector-Project/models/


## Sanity check

val_metrics is already printed above by run_pilot(), but here it is as a dict for quick reference or to compare directly against the Baseline's val_metrics from Baseline_Autoencoder.ipynb.

In [3]:
val_metrics

{'threshold': 0.005,
 'accuracy': 0.992388218807268,
 'precision': 0.9580970592113939,
 'recall': 1.0,
 'f1': 0.9786001717374101,
 'fpr': np.float64(0.009215687696223092),
 'fnr': np.float64(0.0),
 'n': 166978,
 'n_attack': 29061,
 'n_benign': 137917}

In [4]:
from src.evaluate import load_split, context_only_baseline, print_report

train_df = load_split(PROJECT_ROOT / "data" / "processed", "train")
val_df = load_split(PROJECT_ROOT / "data" / "processed", "val")

context_metrics = context_only_baseline(train_df, val_df, target_fpr=0.01)
print_report("Context-only (Destination Port alone)", context_metrics)


=== Context-only (Destination Port alone) ===
  threshold : 0.6736
  accuracy  : 0.8260
  precision : 0.0000
  recall    : 0.0000
  f1        : 0.0000
  fpr       : 0.0000
  fnr       : 1.0000
  n         : 167063
  n_attack  : 29061
  n_benign  : 138002


## Final test-set score (run ONCE, at the end)

Don't run this cell while you're still iterating on the model above — it's meant to be called exactly once, after the model and threshold are frozen, for the real milestone comparison. It will refuse to run twice against the same saved model (see the lock-file check in `src/rf_ids.py`).

In [5]:
# from src.rf_ids import score_test
# test_metrics = score_test(
#     processed_dir=PROJECT_ROOT / "data" / "processed",
#     models_dir=PROJECT_ROOT / "models",
# )